In [ ]:
import torch

# GPU 사용 여부 확인
if torch.cuda.is_available():
    print("GPU를 사용할 수 있습니다!")
    device = torch.device("cuda")
else:
    print("GPU를 사용할 수 없습니다. CPU를 사용합니다.")
    device = torch.device("cpu")


In [ ]:
import json
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import random

# 파일 경로 정의
bert_input_file = "./train_F/bert_input.json"

# JSON 데이터 로드
print("JSON 파일 로드 중...")
with open(bert_input_file, "r") as f:
    data = json.load(f)

# 데이터 준비
print("데이터 준비 중...")
texts = []  # BERT 입력용 텍스트
labels = []  # 레이블

for entry in data:
    # 각 entry에서 sentence와 label 추출
    sentence = entry["sentence"]
    label = entry["label"]
    
    # 리스트에 추가
    texts.append(sentence)
    labels.append(label)

print("데이터 준비 완료!")
print(f"총 데이터 개수: {len(texts)}")


print("학습 데이터와 검증 데이터 분리 중...")
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# 학습 데이터에서 정답 레이블이 1인 데이터 복제
print("정답 레이블이 1인 데이터 복제 중...")
positive_indices = [i for i, label in enumerate(train_labels) if label == 1]
duplicated_texts = []
duplicated_labels = []

for idx in positive_indices:
    for _ in range(10):  # 10배로 복제
        duplicated_texts.append(train_texts[idx])
        duplicated_labels.append(train_labels[idx])

# 기존 학습 데이터에 복제된 데이터 추가
train_texts += duplicated_texts
train_labels += duplicated_labels

# 데이터 순서 섞기
print("학습 데이터 섞기 중...")
combined_data = list(zip(train_texts, train_labels))
random.shuffle(combined_data)
train_texts, train_labels = zip(*combined_data)

# 최종 결과 확인
print(f"학습 데이터 크기: {len(train_texts)}")
print(f"검증 데이터 크기: {len(val_texts)}")
print(f"정답 레이블이 1인 데이터 개수: {train_labels.count(1)}")
print(f"정답 레이블이 0인 데이터 개수: {train_labels.count(0)}")



In [ ]:
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import TrainerCallback

# compute_metrics 함수 정의
def compute_metrics(pred):
    # 예측값과 실제값
    logits = pred.predictions
    labels = pred.label_ids
    predictions = logits.argmax(axis=-1)  # 가장 높은 확률의 클래스 선택

    # 평가 지표 계산
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


class F1CheckpointCallback(TrainerCallback):
    def __init__(self, save_dir="./results", metric_name="f1", greater_is_better=True):
        super().__init__()
        self.save_dir = save_dir
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        if current_metric is None:
            return
        
        # 최초 평가 또는 성능 향상 시 체크포인트 저장
        if self.best_metric is None or (
            self.greater_is_better and current_metric > self.best_metric
        ) or (
            not self.greater_is_better and current_metric < self.best_metric
        ):
            self.best_metric = current_metric
            save_path = os.path.join(self.save_dir, f"best_model_{self.metric_name}.bin")
            kwargs["model"].save_pretrained(save_path)
            print(f"Best {self.metric_name}: {self.best_metric}. Model saved to {save_path}.")


In [ ]:
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

# Tokenizer 준비
print("Tokenizer 준비 중...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 텍스트 토큰화
print("텍스트 토큰화 중...")
train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
val_encodings = tokenizer(val_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

# 확인용 출력
print("텍스트 예시:")
print(f"Train Texts Example: {train_texts[:2]}")
print(f"Train Labels Example: {train_labels[:2]}")

# 모델 초기화
print("모델 초기화 중...")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


In [ ]:
class BERTDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# 데이터셋 생성
print("데이터셋 생성 중...")
train_dataset = BERTDataset(train_encodings, train_labels)
val_dataset = BERTDataset(val_encodings, val_labels)


In [ ]:
# 학습 파라미터 설정
training_args = TrainingArguments(
    output_dir="./results",          # 출력 디렉토리
    evaluation_strategy="steps",    # 평가 전략을 steps로 변경
    save_strategy="steps",          # 저장 전략을 steps로 변경
    save_steps=500,                 # 체크포인트 저장 간격
    eval_steps=500,                 # 평가 간격
    learning_rate=2e-5,             # 학습률
    per_device_train_batch_size=16, # GPU 사용 시 배치 크기
    per_device_eval_batch_size=16,  # GPU 사용 시 배치 크기
    num_train_epochs=10,            # 학습 epoch 수
    weight_decay=0.01,              # 가중치 감소 (L2 정규화)
    logging_dir="./logs",           # 로그 디렉토리
    logging_steps=20,               # 로그 기록 간격
    save_total_limit=2,             # 체크포인트 저장 개수
    fp16=True,                      # 혼합 정밀도 학습 활성화 (GPU 사용 시 유용)
    load_best_model_at_end=True,    # 가장 성능이 좋은 모델을 마지막에 로드
    metric_for_best_model="f1",     # F1 점수를 기준으로 최적 모델 선택
    greater_is_better=True,         # F1 점수는 클수록 좋음
)

# Trainer 초기화
trainer = Trainer(
    model=model,                         # 모델
    args=training_args,                  # 학습 파라미터
    train_dataset=train_dataset,         # 학습 데이터셋
    eval_dataset=val_dataset,            # 검증 데이터셋
    compute_metrics=compute_metrics,     # 평가 메트릭스 함수
    callbacks=[F1CheckpointCallback()]   # 사용자 정의 콜백 추가
)

# 학습 시작
print("모델 학습 시작...")
trainer.train()
print("모델 학습 완료!")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 모델 예측 수행
print("모델 예측 중...")
predictions_output = trainer.predict(val_dataset)

# 예측값과 실제값 추출
logits = predictions_output.predictions
labels = predictions_output.label_ids
predictions = logits.argmax(axis=-1)  # 가장 높은 확률을 가진 클래스 선택

# 수동으로 메트릭 계산
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions, average="weighted")  # 다중 클래스라면 weighted
precision = precision_score(labels, predictions, average="weighted")
recall = recall_score(labels, predictions, average="weighted")

# 결과 출력
print("\nVal 데이터셋 평가 결과:")
print(f"정확도(accuracy): {accuracy:.4f}")
print(f"F1 스코어: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")


In [ ]:
# 파일 경로 정의
bert_input_file = "./test_F/bert_input.json"  # 테스트 데이터 경로

# JSON 데이터 로드
print("JSON 파일 로드 중...")
with open(bert_input_file, "r") as f:
    data = json.load(f)

# 데이터 준비
print("테스트 데이터 준비 중...")
test_texts = []  # BERT 입력용 텍스트
test_labels = []  # 레이블

for entry in data:
    # 각 entry에서 sentence와 label 추출
    sentence = entry["sentence"]
    label = entry["label"]
    
    # 리스트에 추가
    test_texts.append(sentence)
    test_labels.append(label)

print("테스트 데이터 준비 완료!")
print(f"총 테스트 데이터 개수: {len(test_texts)}")

# 데이터 인코딩
print("테스트 데이터 인코딩 중...")
test_encodings = tokenizer(test_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
test_dataset = BERTDataset(test_encodings, test_labels)

print("테스트 데이터셋 생성 완료!")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 모델 예측 수행
print("테스트 데이터셋에 대해 모델 예측 중...")
predictions_output = trainer.predict(test_dataset)  # 테스트 데이터셋 사용

# 예측값과 실제값 추출
logits = predictions_output.predictions
labels = predictions_output.label_ids
predictions = logits.argmax(axis=-1)  # 가장 높은 확률을 가진 클래스 선택

# 메트릭 계산
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions, average="weighted")  # 다중 클래스라면 weighted
precision = precision_score(labels, predictions, average="weighted")
recall = recall_score(labels, predictions, average="weighted")

# 결과 출력
print("\n테스트 데이터셋 평가 결과:")
print(f"정확도(accuracy): {accuracy:.4f}")
print(f"F1 스코어: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")


In [ ]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer
from transformers import Trainer, TrainingArguments
import torch

# Tokenizer 준비
print("Tokenizer 준비 중...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# 텍스트 토큰화
print("텍스트 토큰화 중...")
train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
val_encodings = tokenizer(val_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

# 확인용 출력
print("텍스트 예시:")
print(f"Train Texts Example: {train_texts[:2]}")
print(f"Train Labels Example: {train_labels[:2]}")

# 모델 초기화
print("모델 초기화 중...")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

In [ ]:
class BERTDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# 데이터셋 생성
train_dataset = BERTDataset(train_encodings, train_labels)
val_dataset = BERTDataset(val_encodings, val_labels)


In [ ]:
from collections import Counter

print("Train Dataset Class Distribution:", Counter(train_labels))
print("Validation Dataset Class Distribution:", Counter(val_labels))


In [ ]:
# 학습 파라미터 설정
training_args = TrainingArguments(
    output_dir="./results",          # 출력 디렉토리
    evaluation_strategy="steps",    # 평가 전략을 steps로 변경
    save_strategy="steps",          # 저장 전략을 steps로 변경
    save_steps=500,                 # 체크포인트 저장 간격
    eval_steps=500,                 # 평가 간격
    learning_rate=2e-5,             # 학습률
    per_device_train_batch_size=16, # GPU 사용 시 배치 크기
    per_device_eval_batch_size=16,  # GPU 사용 시 배치 크기
    num_train_epochs=10,            # 학습 epoch 수
    weight_decay=0.01,              # 가중치 감소 (L2 정규화)
    logging_dir="./logs",           # 로그 디렉토리
    logging_steps=20,               # 로그 기록 간격
    save_total_limit=2,             # 체크포인트 저장 개수
    fp16=True,                      # 혼합 정밀도 학습 활성화 (GPU 사용 시 유용)
    load_best_model_at_end=True,    # 가장 성능이 좋은 모델을 마지막에 로드
    metric_for_best_model="f1",     # F1 점수를 기준으로 최적 모델 선택
    greater_is_better=True,         # F1 점수는 클수록 좋음
)

# Trainer 초기화
trainer = Trainer(
    model=model,                         # 모델
    args=training_args,                  # 학습 파라미터
    train_dataset=train_dataset,         # 학습 데이터셋
    eval_dataset=val_dataset,            # 검증 데이터셋
    compute_metrics=compute_metrics,     # 평가 메트릭스 함수
    callbacks=[F1CheckpointCallback()]   # 사용자 정의 콜백 추가
)

# 학습 시작
print("모델 학습 시작...")
trainer.train()
print("모델 학습 완료!")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 모델 예측 수행
print("모델 예측 중...")
predictions_output = trainer.predict(val_dataset)

# 예측값과 실제값 추출
logits = predictions_output.predictions
labels = predictions_output.label_ids
predictions = logits.argmax(axis=-1)  # 가장 높은 확률을 가진 클래스 선택

# 수동으로 메트릭 계산
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions, average="weighted")  # 다중 클래스라면 weighted
precision = precision_score(labels, predictions, average="weighted")
recall = recall_score(labels, predictions, average="weighted")

# 결과 출력
print("\nVal 데이터셋 평가 결과:")
print(f"정확도(accuracy): {accuracy:.4f}")
print(f"F1 스코어: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")


In [ ]:
# 파일 경로 정의
bert_input_file = "./test_F/bert_input.json"  # 테스트 데이터 경로

# JSON 데이터 로드
print("JSON 파일 로드 중...")
with open(bert_input_file, "r") as f:
    data = json.load(f)

# 데이터 준비
print("테스트 데이터 준비 중...")
test_texts = []  # BERT 입력용 텍스트
test_labels = []  # 레이블

for entry in data:
    # 각 entry에서 sentence와 label 추출
    sentence = entry["sentence"]
    label = entry["label"]
    
    # 리스트에 추가
    test_texts.append(sentence)
    test_labels.append(label)

print("테스트 데이터 준비 완료!")
print(f"총 테스트 데이터 개수: {len(test_texts)}")

# 데이터 인코딩
print("테스트 데이터 인코딩 중...")
test_encodings = tokenizer(test_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
test_dataset = BERTDataset(test_encodings, test_labels)

print("테스트 데이터셋 생성 완료!")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 모델 예측 수행
print("테스트 데이터셋에 대해 모델 예측 중...")
predictions_output = trainer.predict(test_dataset)  # 테스트 데이터셋 사용

# 예측값과 실제값 추출
logits = predictions_output.predictions
labels = predictions_output.label_ids
predictions = logits.argmax(axis=-1)  # 가장 높은 확률을 가진 클래스 선택

# 메트릭 계산
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions, average="weighted")  # 다중 클래스라면 weighted
precision = precision_score(labels, predictions, average="weighted")
recall = recall_score(labels, predictions, average="weighted")

# 결과 출력
print("\n테스트 데이터셋 평가 결과:")
print(f"정확도(accuracy): {accuracy:.4f}")
print(f"F1 스코어: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
